# Survey Health

This notebook reads the cached TeamDynamix survey model from DuckDB and frames the results for executive review: coverage, sentiment, time trends, team/service patterns, and low-score comment review.

In [1]:
from pathlib import Path

import duckdb
import pandas as pd

import plotly.express as px

from dynamix_manager.metrics import summarize_survey_health

db_path = Path('/Users/micahcooper/dynamix-manager/data/analytics.duckdb')
connection = duckdb.connect(str(db_path), read_only=True)


In [2]:
survey_health = connection.execute(
    "select * from ticket_linked_surveys"
).fetchdf()
for column in ['survey_completed_at', 'created_at', 'responded_at', 'resolved_at']:
    if column in survey_health.columns:
        survey_health[column] = pd.to_datetime(survey_health[column], utc=True, errors='coerce')
survey_health.head()

,response_id,ticket_id,survey_requested_at,survey_completed_at,satisfaction_label,comment_text,ticket_title,status_name,status_class,type_name,...,respond_by_at,resolve_by_at,is_sla_violated,is_sla_respond_by_violated,is_sla_resolve_by_violated,ticket_app_id,ticket_app_name,ticket_linked,response_time_hours,resolution_time_hours
0,1069436,5103123.0,2018-03-06T15:58:52.31Z,2018-03-06 16:11:34.933000+00:00,Very Satisfied,Dell Laptop in HR Conference Room not connecti...,Dell Laptop in HR Conference Room not connecti...,Closed,3.0,Office Devices,...,0001-01-01T00:00:00,2018-03-08T15:48:00Z,False,False,False,634.0,InfoTech Tickets,True,0.0,0.0
1,1070810,5105493.0,2018-03-06T21:54:38.267Z,2018-03-07 13:21:07.677000+00:00,Very Satisfied,Error message in VOUM of Colleague,Error message in VOUM of Colleague,Closed,3.0,Administrative Systems,...,0001-01-01T00:00:00,2018-03-08T17:59:00Z,False,False,False,634.0,InfoTech Tickets,True,NaN,NaN
2,1070915,5109784.0,2018-03-06T22:17:49.163Z,2018-03-06 22:22:33.827000+00:00,Very Satisfied,Direct call for Micah Cooper,Direct call for Micah Cooper,Closed,3.0,Generic Submission,...,0001-01-01T00:00:00,2018-03-08T22:00:00Z,False,False,False,634.0,InfoTech Tickets,True,NaN,NaN
3,1071315,5107948.0,2018-03-07T13:36:41.54Z,2018-03-16 15:36:10.047000+00:00,Satisfied,New employee's ID# may have been deleted from RE,New employee's ID# may have been deleted from RE,Closed,3.0,Administrative Systems,...,0001-01-01T00:00:00,2018-03-08T20:13:00Z,False,False,False,634.0,InfoTech Tickets,True,NaN,NaN
4,1071623,5110052.0,2018-03-07T15:19:03.85Z,2018-03-07 15:24:06.380000+00:00,Very Satisfied,The Problem is NOT in a classroom it is in the...,The Problem is NOT in a classroom it is in the...,Closed,3.0,Office Devices,...,0001-01-01T00:00:00,2018-03-08T22:00:00Z,False,False,False,634.0,InfoTech Tickets,True,NaN,NaN


## Executive Summary

In [3]:
summary = summarize_survey_health(survey_health)
pd.DataFrame(
    [
        {'metric': 'Responses', 'value': summary['total_responses']},
        {'metric': 'Linked responses', 'value': summary['linked_responses']},
        {'metric': 'Link rate', 'value': summary['linked_responses'] / summary['total_responses'] if summary['total_responses'] else 0},
        {'metric': 'Comment rate', 'value': summary['comment_rate']},
        {'metric': 'Average score', 'value': summary['average_score']},
        {'metric': 'Negative response rate', 'value': summary['negative_response_rate']},
    ]
)

,metric,value
0,Responses,5843.000000
1,Linked responses,5843.000000
2,Link rate,1.000000
3,Comment rate,1.000000
4,Average score,4.815323
5,Negative response rate,0.000000


## Satisfaction Mix

In [4]:
satisfaction_mix = (
    survey_health.groupby('satisfaction_label', dropna=False)
    .size()
    .sort_values(ascending=False)
    .to_frame('responses')
)
satisfaction_mix

,responses
satisfaction_label,
Very Satisfied,4512
Satisfied,1022
Very Unsatisfied,182
Unsatisfied,69
NaN,58


In [5]:
satisfaction_mix_chart = satisfaction_mix.reset_index(names='satisfaction_label')
px.bar(
    satisfaction_mix_chart,
    x='satisfaction_label',
    y='responses',
    title='Survey Satisfaction Mix',
).show()


## Monthly Trend

In [6]:
monthly_trend = (
    survey_health.assign(month=survey_health['survey_completed_at'].dt.to_period('M').dt.to_timestamp())
    .groupby('month', dropna=False)
    .agg(
        responses=('response_id', 'count'),
        linked_responses=('ticket_linked', 'sum'),
        avg_score=('satisfaction_label', lambda s: s.map({'Very Dissatisfied': 1, 'Dissatisfied': 2, 'Satisfied': 4, 'Very Satisfied': 5}).mean()),
        negative_rate=('satisfaction_label', lambda s: s.isin(['Very Dissatisfied', 'Dissatisfied']).mean()),
    )
    .sort_index()
)
monthly_trend.tail(12)

/var/folders/5d/hx576nj50ss0443h526w89440000gq/T/ipykernel_34779/2806803918.py:2: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  survey_health.assign(month=survey_health['survey_completed_at'].dt.to_period('M').dt.to_timestamp())


,responses,linked_responses,avg_score,negative_rate
month,,,,
2025-05-01,20,20,4.842105,0.0
2025-06-01,20,20,4.789474,0.0
2025-07-01,25,25,4.863636,0.0
2025-08-01,120,120,4.817391,0.0
2025-09-01,66,66,4.812500,0.0
2025-10-01,52,52,4.877551,0.0
2025-11-01,30,30,4.800000,0.0
2025-12-01,30,30,4.833333,0.0
2026-01-01,47,47,4.795455,0.0


In [7]:
monthly_trend_chart = monthly_trend.reset_index()
px.line(
    monthly_trend_chart,
    x='month',
    y=['responses', 'linked_responses'],
    title='Monthly Survey Volume',
).show()


## Team Breakdown

In [8]:
team_summary = (
    survey_health.loc[survey_health['ticket_linked']]
    .groupby('team_name', dropna=False)
    .agg(
        responses=('response_id', 'count'),
        avg_score=('satisfaction_label', lambda s: s.map({'Very Dissatisfied': 1, 'Dissatisfied': 2, 'Satisfied': 4, 'Very Satisfied': 5}).mean()),
        negative_rate=('satisfaction_label', lambda s: s.isin(['Very Dissatisfied', 'Dissatisfied']).mean()),
        avg_response_hours=('response_time_hours', 'mean'),
        avg_resolution_hours=('resolution_time_hours', 'mean'),
    )
    .sort_values(['responses', 'avg_score'], ascending=[False, False])
)
team_summary.head(15)

,responses,avg_score,negative_rate,avg_response_hours,avg_resolution_hours
team_name,,,,,
Tech Services,1938,4.849067,0.0,NaN,NaN
User Services,1220,4.805699,0.0,0.0,0.0
Information Systems & Services (ISS),922,4.748874,0.0,NaN,NaN
TechStop,506,4.834711,0.0,NaN,NaN
Network Infrastructure Services,478,4.794118,0.0,NaN,NaN
Network Students,338,4.790123,0.0,NaN,NaN
Tech Services Students,148,4.843972,0.0,NaN,NaN
User Services Staff,146,4.892086,0.0,NaN,NaN
Classroom & Lab,110,4.836538,0.0,NaN,NaN


In [9]:
team_chart = team_summary.reset_index().head(10)
px.bar(
    team_chart,
    x='team_name',
    y='responses',
    color='avg_score',
    title='Top Teams by Survey Response Volume',
).show()


## Service Breakdown

In [10]:
service_summary = (
    survey_health.loc[survey_health['ticket_linked']]
    .groupby('service_name', dropna=False)
    .agg(
        responses=('response_id', 'count'),
        avg_score=('satisfaction_label', lambda s: s.map({'Very Dissatisfied': 1, 'Dissatisfied': 2, 'Satisfied': 4, 'Very Satisfied': 5}).mean()),
        negative_rate=('satisfaction_label', lambda s: s.isin(['Very Dissatisfied', 'Dissatisfied']).mean()),
    )
    .sort_values('responses', ascending=False)
)
service_summary.head(20)

,responses,avg_score,negative_rate
service_name,,,
Office Devices: Ask for Help / Report a Problem,1038,4.861538,0.0
❓Request help with software,582,4.805405,0.0
IT Internal: General Ticket Submission,506,4.797468,0.0
Classrooms & Labs: Ask for Help / Report a Problem,466,4.820046,0.0
Personal Devices: Ask for Help,423,4.819753,0.0
None,385,4.789041,0.0
📋 Register my Wi-Fi device,293,4.797872,0.0
Classrooms & Labs: Request a Setup,281,4.901515,0.0
Accounts & Access: Request a change in access rights,250,4.795833,0.0


## Time-To-Serve by Satisfaction

In [11]:
time_to_serve = (
    survey_health.groupby('satisfaction_label', dropna=False)
    .agg(
        responses=('response_id', 'count'),
        avg_response_hours=('response_time_hours', 'mean'),
        avg_resolution_hours=('resolution_time_hours', 'mean'),
    )
    .sort_values('responses', ascending=False)
)
time_to_serve

,responses,avg_response_hours,avg_resolution_hours
satisfaction_label,,,
Very Satisfied,4512,0.0,0.0
Satisfied,1022,NaN,NaN
Very Unsatisfied,182,NaN,NaN
Unsatisfied,69,NaN,NaN
NaN,58,NaN,NaN


## Low-Score Review

In [12]:
low_score_comments = survey_health.loc[
    survey_health['satisfaction_label'].isin(['Very Dissatisfied', 'Dissatisfied']),
    ['survey_completed_at', 'team_name', 'service_name', 'assignee_name', 'response_time_hours', 'resolution_time_hours', 'comment_text'],
].sort_values('survey_completed_at', ascending=False)
low_score_comments.head(25)

,survey_completed_at,team_name,service_name,assignee_name,response_time_hours,resolution_time_hours,comment_text


## Recent Comment Review

In [13]:
recent_comments = survey_health.loc[
    survey_health['comment_text'].astype('string').fillna('').str.strip() != '',
    ['survey_completed_at', 'satisfaction_label', 'team_name', 'service_name', 'comment_text'],
].sort_values('survey_completed_at', ascending=False).head(25)
recent_comments

,survey_completed_at,satisfaction_label,team_name,service_name,comment_text
5839,2026-03-30 12:49:03.717000+00:00,NaN,User Services,❓ Ask a Question / Request Help,Action Required: Update Duo Mobile App
5840,2026-03-30 02:44:17.123000+00:00,NaN,Tech Services,IT Internal: General Ticket Submission,SolidWorks - 3D Modeling Software
5842,2026-03-29 21:48:12.243000+00:00,NaN,Tech Services,❓Request help with software,"KiCad 9.0, Reinstall"
5841,2026-03-28 17:47:41.793000+00:00,NaN,Tech Services,❓ Ask a Question / Request Help,question
5838,2026-03-27 19:08:00.440000+00:00,NaN,Tech Services Students,Office Devices: Ask for Help / Report a Problem,My laptop (apple-mac) is having a hard time co...
5837,2026-03-27 18:18:39.367000+00:00,NaN,TechStop,Personal Devices: Ask for Help,laptop won't start
5836,2026-03-27 16:21:15.423000+00:00,NaN,User Services,❓ Ask a Question / Request Help,Set up Duo Mobile
5835,2026-03-26 00:51:47.193000+00:00,NaN,Network Students,📋 Register my Wi-Fi device,Manual Registration Request
5833,2026-03-25 21:37:52.973000+00:00,NaN,Tech Services,None,Monitors
5834,2026-03-25 13:07:19.120000+00:00,NaN,Tech Services,Office Devices: Ask for Help / Report a Problem,Extra Monitor has no power
